# 遷移型の依存構造解析の動きを追う (arc-standard)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_parsing/transition_parsing.ipynb)

NLP クックブック 13-12 ／ 構文解析

スタックとバッファを使って、左から 1 語ずつ読みながら依存構造木を組み立てる「遷移型解析」の仕組みを、手を動かして理解したい。

In [1]:
def oracle(words, heads):
    """正解の木 (heads[i] = i 番目の語の主辞。0 が ROOT、語は 1 始まり) を作るための遷移の列を求める"""
    n = len(words)
    n_children = [sum(1 for h in heads if h == i) for i in range(n + 1)]   # まだ付けていない子の数
    stack, buffer, steps, arcs = [0], list(range(1, n + 1)), [], []
    name = lambda i: "ROOT" if i == 0 else words[i - 1]
    while buffer or len(stack) > 1:
        action = "SHIFT"
        if len(stack) >= 2:
            s1, s0 = stack[-2], stack[-1]
            if s1 != 0 and heads[s1 - 1] == s0 and n_children[s1] == 0:
                action = "LEFT-ARC"       # スタックの 2 番目が、一番上の語に係る
            elif heads[s0 - 1] == s1 and n_children[s0] == 0:
                action = "RIGHT-ARC"      # スタックの一番上が、2 番目の語に係る
        if action == "SHIFT" and not buffer:
            raise ValueError("この木は arc-standard では作れません (交差する係り受けがある)")
        steps.append((action, [name(i) for i in stack], [name(i) for i in buffer]))
        if action == "SHIFT":
            stack.append(buffer.pop(0))
        elif action == "LEFT-ARC":
            arcs.append((name(s0), name(s1))); n_children[s0] -= 1; del stack[-2]
        else:
            arcs.append((name(s1), name(s0))); n_children[s1] -= 1; stack.pop()
    return steps, arcs

words = ["太郎", "が", "赤い", "本", "を", "読ん", "だ"]
heads = [6, 1, 4, 6, 4, 0, 6]          # 太郎→読ん, が→太郎, 赤い→本, 本→読ん, を→本, 読ん→ROOT, だ→読ん
steps, arcs = oracle(words, heads)
for action, stack, buffer in steps:
    print(f"{action:10} スタック: {' '.join(stack):24} バッファ: {' '.join(buffer)}")
print("作られた係り受け (主辞 → 従属語):", arcs)

SHIFT      スタック: ROOT                     バッファ: 太郎 が 赤い 本 を 読ん だ
SHIFT      スタック: ROOT 太郎                  バッファ: が 赤い 本 を 読ん だ
RIGHT-ARC  スタック: ROOT 太郎 が                バッファ: 赤い 本 を 読ん だ
SHIFT      スタック: ROOT 太郎                  バッファ: 赤い 本 を 読ん だ
SHIFT      スタック: ROOT 太郎 赤い               バッファ: 本 を 読ん だ
LEFT-ARC   スタック: ROOT 太郎 赤い 本             バッファ: を 読ん だ
SHIFT      スタック: ROOT 太郎 本                バッファ: を 読ん だ
RIGHT-ARC  スタック: ROOT 太郎 本 を              バッファ: 読ん だ
SHIFT      スタック: ROOT 太郎 本                バッファ: 読ん だ
LEFT-ARC   スタック: ROOT 太郎 本 読ん             バッファ: だ
LEFT-ARC   スタック: ROOT 太郎 読ん               バッファ: だ
SHIFT      スタック: ROOT 読ん                  バッファ: だ
RIGHT-ARC  スタック: ROOT 読ん だ                バッファ: 
RIGHT-ARC  スタック: ROOT 読ん                  バッファ: 
作られた係り受け (主辞 → 従属語): [('太郎', 'が'), ('本', '赤い'), ('本', 'を'), ('読ん', '本'), ('読ん', '太郎'), ('読ん', 'だ'), ('ROOT', '読ん')]


In [2]:
# 交差する係り受け (非交差性の違反) があるかを調べる
def is_projective(heads):
    arcs = [(min(i, h), max(i, h)) for i, h in enumerate(heads, 1)]
    return not any(a < c < b < d for a, b in arcs for c, d in arcs)

print(is_projective(heads))
# "A hearing is scheduled on the issue today": on(5) が hearing(2) に係り、is scheduled をまたぐ
en_words = ["A", "hearing", "is", "scheduled", "on", "the", "issue", "today"]
en_heads = [2, 4, 4, 0, 7, 7, 2, 4]
print(is_projective(en_heads))
try:
    oracle(en_words, en_heads)
except ValueError as e:
    print(e)

True
False
この木は arc-standard では作れません (交差する係り受けがある)


- 長さ n の文は、ちょうど 2n 回の遷移 (SHIFT が n 回、ARC が n 回) で解析できる。各時点でどの遷移を選ぶかを分類器で予測するのが遷移型パーザで、spaCy の parser もこの方式。
- 学習時は、ここで求めた「正解の遷移列」を教師データにする (スタックとバッファの状態 → 次の遷移)。
- 従属語をすべて付け終えてからでないと RIGHT-ARC できない (`n_children` の確認)。早く付けてしまうと、その語の子を付けられなくなる。
- 素の arc-standard は交差する係り受けを作れない。日本語の係り受けはほぼ交差しないが、チェコ語やドイツ語などでは無視できない頻度で現れるので、SWAP 遷移の追加などの拡張が使われる。